# 8-0(참고). SV 설명 (직접구현)

SV(Stochastic Volatility)는 이 프로젝트가 직접 구현했다. 왜 직접 만들었는지, 어떤 기법인지, 겪은 문제를 정리한다.

## 6-1. SV vs GARCH

```
GARCH: σ_t² = ω + α ε_(t-1)² + β σ_(t-1)²              -- 어제 정보로 오늘 σ_t²가 정확히 정해짐
SV:    h_t = μ + φ(h_(t-1) - μ) + η_t,  η_t ~ N(0, σ_η²)  -- h_t = log σ_t², 자기만의 노이즈를 가짐
```

SV는 변동성 자체가 우연히 흔들린다(**변동성의 변동성 σ_η**). 이 개념은 SV에만 있다.

## 6-2. 왜 직접 만들었나

Python에는 GARCH의 `arch`처럼 SV의 표준 라이브러리가 없다. R의 `stochvol`은 베이지안 MCMC라 MLE 계열인 이 프로젝트와 방식이 다르다. 그래서 **Harvey·Ruiz·Shephard(1994)의 QML** 기법을 직접 구현했다.

## 6-3. QML -- 칼만필터로 푸는 방법

h_t는 관측할 수 없는 잠재변수다. QML의 트릭:

1. 잔차를 제곱해 로그를 취한다: `z_t = log(ε_t²) = h_t + log(u_t²)`
2. `log(u_t²)`(log-카이제곱분포)를 **정규분포로 근사**한다. 근사의 평균/분산이 `XI_MEAN = -1.2704`, `XI_VAR = π²/2`다.
3. 그러면 선형-가우시안 상태공간모형이 되어 **칼만필터**로 h_t를 필터링하고 우도를 계산할 수 있다.

```python
def sv_qml_negloglik(z_centered, phi, sigma_eta2):
    h_pred, P_pred, loglik = 0.0, sigma_eta2 / (1 - phi**2), 0.0
    for z in z_centered:
        F = P_pred + XI_VAR; v = z - h_pred
        loglik += -0.5 * (np.log(2*np.pi*F) + v**2/F)
        K = P_pred / F
        h_filt, P_filt = h_pred + K*v, P_pred*(1-K)
        h_pred, P_pred = phi*h_filt, phi**2*P_filt + sigma_eta2
    return -loglik
```

핵심 파라미터는 `phi`(로그변동성 지속성, GARCH의 α+β와 비슷)와 `sigma_eta2`(변동성의 변동성)다.

## 6-4. 동시최적화로 확장

원래 QML은 잔차가 이미 주어졌다고 전제한다(ARIMA로 먼저 잔차를 구하는 2단계). 이 프로젝트는 **AR계수까지 한 목적함수에서 같이 최적화**하도록 확장했다.

```python
def joint_sv_negloglik(params, y, p):
    ar_coefs = params[:p]                       # AR계수도 파라미터 벡터 안에 포함
    phi, sigma_eta2 = MAX_PHI * np.tanh(params[p]), np.exp(params[p+1]) ** 2
    resid = [y[t] - sum(ar_coefs[i]*y[t-1-i] for i in range(p)) for t in range(p, n)]
    z = np.log(np.array(resid)**2 + 1e-8)
    return sv_qml_negloglik(z - z.mean(), phi, sigma_eta2)
```

이 때문에 2단계(`0922/7-1`, 점예측 100% 동일)와 달리 동시최적화(`8-2`)는 점예측이 달라진다.

## 6-5. 개발 중 발견하고 고친 문제

1. **절편 폭발**: 절편을 자유파라미터로 두면 d=0에서 AR계수와 서로 보정하며 절편이 수십억으로 튀었다. -> 절편 없는 AR로 적합. (처음엔 평균 제거 후 적합했으나 추정을 불안정하게 하고 다른 모델과 조건이 달라 `8-6(참고)`에서 제거)
2. **phi≈1 불안정**: phi가 1에 가까우면 다단계 분산이 `exp()`로 폭발했다(CRPS 수만). 베타 사전분포는 효과가 없어 `phi = 0.97 * tanh(...)` 상한으로 해결(민감도 6-6).
3. **p를 AIC로 고르면 무효**: 조건부우도라 상한에 쏠렸다(`8-3`, `8-4` 7절). -> p는 그리드서치로 확정.
4. **AR 안정성 체크 부재**: 발산하는 AR계수로 다단계 예측하면 폭발했다. -> 컴패니언 행렬 고유값(`is_ar_stable`)으로 불안정한 후보를 버림.

## 6-3-1. `stochvolmodels` 패키지는 왜 못 쓰나

PyPI의 `stochvolmodels`(Artur Sepp)는 **옵션 가격결정용**이라 전제가 다르다.

| | `stochvolmodels` | 이 프로젝트 |
|---|---|---|
| 모델 | Heston 등 연속시간 SDE | 이산시간 AR(1) 로그변동성 |
| 변동성 | 이미 관측된 시계열(내재변동성 등)에 맞춤 | 관측 안 되는 잠재변수를 수익률로 역추정 |
| 용도 | 옵션 가격·캘리브레이션 | `X3_l1` 하나로 미래 σ 예측 |

이름만 같고 데이터 전제와 모델이 달라 재사용할 수 없다.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy.optimize import minimize

raw = pd.read_csv("../../data/gold/gold_major_variables.csv")
raw = raw[raw["quarter"].notna()].reset_index(drop=True)
vals = raw[["quarter", "X3_l1"]].dropna()["X3_l1"].values.astype(float)
train = vals[:35]
diff_train = np.diff(train, n=1)

XI_MEAN, XI_VAR = -1.2704, np.pi ** 2 / 2
MAX_PHI = 0.97


def is_ar_stable(ar_coefs):
    p = len(ar_coefs)
    if p == 0:
        return True
    companion = np.zeros((p, p))
    companion[0, :] = ar_coefs
    if p > 1:
        companion[1:, :-1] = np.eye(p - 1)
    return np.max(np.abs(np.linalg.eigvals(companion))) < 0.999


def sv_qml_negloglik(z_centered, phi, sigma_eta2):
    P0 = sigma_eta2 / (1 - phi ** 2) if abs(phi) < 0.999 else sigma_eta2 * 100
    h_pred, P_pred, loglik = 0.0, P0, 0.0
    for z in z_centered:
        F = P_pred + XI_VAR
        v = z - h_pred
        loglik += -0.5 * (np.log(2 * np.pi * F) + v ** 2 / F)
        K = P_pred / F
        h_filt, P_filt = h_pred + K * v, P_pred * (1 - K)
        h_pred, P_pred = phi * h_filt, phi ** 2 * P_filt + sigma_eta2
    return -loglik


def joint_sv_negloglik(params, y, p):
    ar_coefs = params[:p]
    phi = MAX_PHI * np.tanh(params[p])
    sigma_eta2 = np.exp(params[p + 1]) ** 2
    if not is_ar_stable(ar_coefs):
        return 1e10
    n = len(y)
    resid = np.array([y[t] - sum(ar_coefs[i] * y[t - 1 - i] for i in range(p)) for t in range(p, n)])
    z = np.log(resid ** 2 + 1e-8)
    return sv_qml_negloglik(z - z.mean(), phi, sigma_eta2)


p = 1
x0 = [0.0] * p + [1.5, np.log(0.3)]
res = minimize(joint_sv_negloglik, x0, args=(diff_train, p), method="Nelder-Mead", options={"maxiter": 3000})

ar_coef = res.x[0]
phi = MAX_PHI * np.tanh(res.x[p])
sigma_eta = np.exp(res.x[p + 1])
print(f"AR(1) 동시최적화 결과: AR계수={ar_coef:.4f}, phi(로그변동성 지속성)={phi:.4f}, sigma_eta(변동성의 변동성)={sigma_eta:.4f}")
print(f"AR 안정성 통과: {is_ar_stable(res.x[:p])}")


AR(1) 동시최적화 결과: AR계수=-0.0438, phi(로그변동성 지속성)=0.7047, sigma_eta(변동성의 변동성)=0.3216
AR 안정성 통과: True


## 6-6. 파라미터는 어떻게 찾나

`fit_sv_joint`가 매 origin마다 수행한다: (1) 훈련 데이터를 그대로 사용, (2) `[AR계수, phi 변환값, log σ_η]`를 잡아 `phi = 0.97·tanh(x)`, `σ_η = exp(x)`로 범위 보장, (3) QML 음의 로그우도를 Nelder-Mead(최대 3000회)로 최소화, (4) AR 안정성을 통과하지 못하면 해를 버림.

| 구분 | 내용 |
|---|---|
| 자동 추정 | AR계수, phi, σ_η |
| 사람이 고정 | SV(1) 차수, `MAX_PHI=0.97`, `XI_MEAN`/`XI_VAR`, 시작점 1개 |
| 그리드서치 | d, p (`8-4`) |

**한계**: 시작점이 1개라 코너해가 생길 수 있으나, 현재 파이프라인은 22개 origin 중 0개(`8-2` 5절)라 멀티스타트가 필요 없다. (초기 버전은 5/22, `결함판(참고용)/8-2`)

### `MAX_PHI=0.97`은 고정해도 되나

phi≈1의 분산 폭발을 막으려고 넣은 값이고 검증한 적이 없다. 상한을 바꿔도 성능이 변하는지 d=1, p=1 SV동시로 확인했다.

**결과**(`8-2` 7절에서 수행): `MAX_PHI` 0.95~0.99에서 결과가 같고(경계 도달 0/22), 0.90에서는 9/22가 상한에 걸리지만 성능은 같다(MSE 1.052, CRPS 0.577). AR계수≈0도 상한과 무관하다(SV 우도의 구조적 특징). **0.97 고정은 무방하다.**

## 요약

| | ARIMA·GARCH | SV |
|---|---|---|
| 도구 | 라이브러리(`pmdarima`, `arch`) | 직접 구현 |
| 이유 | 성숙한 라이브러리가 있음 | Python에 표준 SV 도구가 없음 |
| 핵심 기법 | MLE | QML + 칼만필터 |
| 동시최적화 | GARCH는 원래 평균+분산 동시추정 | AR계수까지 포함하도록 직접 확장 |